# MuseTalk 1.5 — T4 Colab worker (FIXED)

**This notebook is valid JSON and does not require the Hugging Face `hf` CLI.**

Run the cells top-to-bottom. The worker stops immediately if CUDA/T4 is unavailable or a required model is missing.

In [ ]:
# 1) Create a clean Python 3.10 environment and verify the T4
import os, sys, shutil, subprocess
from pathlib import Path

# Colab's system Python can change; MuseTalk runs inside this isolated 3.10 venv.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     text=True, capture_output=True, check=False)
print("Visible GPU:", gpu.stdout.strip() or "NONE")
if not gpu.stdout.strip():
    raise RuntimeError("STOP: No NVIDIA GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU.")

MT = Path("/content/MuseTalk")
VENV = Path("/content/musetalk310")

if not MT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/TMElyralab/MuseTalk.git", str(MT)], check=True)

if VENV.exists() and not (VENV / "bin/python").exists():
    shutil.rmtree(VENV)

if not VENV.exists():
    subprocess.run(["uv", "venv", "--python", "3.10", "--seed", str(VENV)], check=True)

PY = str(VENV / "bin/python")
UV = ["uv", "pip", "install", "--python", PY]

# Install the exact CUDA 11.8 PyTorch stack first.
subprocess.run(UV + ["pip==24.0", "setuptools==69.5.1", "wheel==0.43.0"], check=True)
subprocess.run(UV + ["torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2",
                     "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)

probe = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"],
    text=True, capture_output=True, check=True
)
print(probe.stdout)
lines = probe.stdout.strip().splitlines()
if len(lines) < 2 or lines[1].strip() != "True":
    raise RuntimeError("STOP: CUDA is not visible inside the MuseTalk Python 3.10 environment.")

print("T4 CUDA environment is ready.")


In [ ]:
# 2) Install MuseTalk dependencies (NO `hf` executable)
import subprocess
from pathlib import Path

PY = "/content/musetalk310/bin/python"
MT = Path("/content/MuseTalk")
UV = ["uv", "pip", "install", "--python", PY]

# MuseTalk's pinned requirements. Reinstalling this file is safe because the
# critical torch version was already pinned above.
subprocess.run(UV + ["-r", str(MT / "requirements.txt")], check=True)

# Make sure the Python API is present; the notebook never calls `hf`.
subprocess.run(UV + ["huggingface_hub==0.30.2", "gdown"], check=True)

# Required MMLab packages.
MIM = "/content/musetalk310/bin/mim"
for pkg in ["mmengine", "mmcv==2.0.1", "mmdet==3.1.0", "mmpose==1.1.0"]:
    p = subprocess.run([MIM, "install", pkg], text=True, capture_output=True)
    print(p.stdout)
    if p.returncode:
        print(p.stderr)
        raise RuntimeError("MMLab installation failed: " + pkg)

check = subprocess.run(
    [PY, "-c", "import torch, diffusers, transformers, mmcv, mmdet, mmpose; print('imports OK'); print(torch.cuda.is_available())"],
    text=True, capture_output=True
)
print(check.stdout)
if check.returncode != 0 or check.stdout.strip().splitlines()[-1] != "True":
    raise RuntimeError("MuseTalk Python environment is not healthy:\n" + check.stderr)

print("MuseTalk dependencies installed.")


In [ ]:
# 3) Download all required weights (Hugging Face Python API — NO `hf` CLI)
import subprocess
from pathlib import Path

PY = "/content/musetalk310/bin/python"
GDOWN = "/content/musetalk310/bin/gdown"
MODELS = Path("/content/MuseTalk/models")
MODELS.mkdir(parents=True, exist_ok=True)

download_code = r"""
from huggingface_hub import snapshot_download
from pathlib import Path

def hf_files(repo, local_dir, patterns):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {repo} -> {local_dir}")
    snapshot_download(repo_id=repo, local_dir=str(local_dir), allow_patterns=patterns)

hf_files("TMElyralab/MuseTalk", "/content/MuseTalk/models", [
    "musetalkV15/unet.pth", "musetalkV15/musetalk.json"
])
hf_files("stabilityai/sd-vae-ft-mse", "/content/MuseTalk/models/sd-vae", [
    "config.json", "diffusion_pytorch_model.bin"
])
hf_files("openai/whisper-tiny", "/content/MuseTalk/models/whisper", [
    "config.json", "preprocessor_config.json", "pytorch_model.bin"
])
hf_files("yzd-v/DWPose", "/content/MuseTalk/models/dwpose", [
    "dw-ll_ucoco_384.pth"
])
"""
print("Downloading Hugging Face weights...")
p = subprocess.run([PY, "-c", download_code], text=True)
if p.returncode:
    raise RuntimeError("Hugging Face model download failed.")

FACE = MODELS / "face-parse-bisent"
FACE.mkdir(parents=True, exist_ok=True)
face_pth = FACE / "79999_iter.pth"
resnet_pth = FACE / "resnet18-5c106cde.pth"

if not face_pth.exists() or face_pth.stat().st_size < 100_000:
    print("Downloading face parsing weights...")
    subprocess.run([GDOWN, "--id", "154JgKpzCPW82qINcVieuPH3fZ2e0P812", "-O", str(face_pth)], check=True)

if not resnet_pth.exists() or resnet_pth.stat().st_size < 1_000_000:
    print("Downloading ResNet18...")
    subprocess.run(["curl", "-L", "--fail", "--retry", "5", "-o", str(resnet_pth),
                    "https://download.pytorch.org/models/resnet18-5c106cde.pth"], check=True)

required = [
    MODELS/"musetalkV15/unet.pth", MODELS/"musetalkV15/musetalk.json",
    MODELS/"sd-vae/config.json", MODELS/"sd-vae/diffusion_pytorch_model.bin",
    MODELS/"whisper/config.json", MODELS/"whisper/preprocessor_config.json", MODELS/"whisper/pytorch_model.bin",
    MODELS/"dwpose/dw-ll_ucoco_384.pth", FACE/"79999_iter.pth", FACE/"resnet18-5c106cde.pth",
]
for f in required:
    if not f.exists() or f.stat().st_size < 1000:
        raise RuntimeError(f"Missing/incomplete model: {f} ({f.stat().st_size if f.exists() else 0} bytes)")

print("\nALL MODEL FILES READY.")


In [ ]:
# 4) Upload the approved singer image/video and successful ACE-Step audio
from google.colab import files
from pathlib import Path
import shutil, subprocess

OUT = Path("/content/musetalk_output")
OUT.mkdir(parents=True, exist_ok=True)

print("Upload the APPROVED singer image/video:")
up = files.upload()
if not up:
    raise RuntimeError("No singer file uploaded.")
avatar_name = next(iter(up))
avatar_input = OUT / avatar_name
shutil.move(avatar_name, avatar_input)

print("Upload the successful ACE-Step bhajan MP3/WAV:")
up = files.upload()
if not up:
    raise RuntimeError("No audio file uploaded.")
audio_name = next(iter(up))
audio_input = OUT / audio_name
shutil.move(audio_name, audio_input)

avatar_source = OUT / "avatar_source.png"
audio_wav = OUT / "audio.wav"

subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(avatar_input),
                "-frames:v", "1", "-vf", "scale=512:-2", str(avatar_source)], check=True)
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(audio_input),
                "-ar", "16000", "-ac", "1", str(audio_wav)], check=True)

print("Singer:", avatar_source)
print("Audio :", audio_wav)
print("Inputs are ready.")


In [ ]:
# 5) Run MuseTalk 1.5 on the T4
import os, subprocess
from pathlib import Path

MT = Path("/content/MuseTalk")
OUT = Path("/content/musetalk_output")
PY = "/content/musetalk310/bin/python"

cfg = MT / "configs/inference/colab_bhajan.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text(
    "bhajan_test:\n"
    f"  video_path: \"{OUT / 'avatar_source.png'}\"\n"
    f"  audio_path: \"{OUT / 'audio.wav'}\"\n"
    "  result_name: \"bhajan_lipsync.mp4\"\n"
)

probe = subprocess.run(
    [PY, "-c", "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"],
    text=True, capture_output=True
)
print(probe.stdout)
if probe.returncode != 0 or probe.stdout.splitlines()[0].strip() != "True":
    raise RuntimeError("STOP: T4 CUDA is not available:\n" + probe.stderr)

result_dir = OUT / "result"
result_dir.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTHONPATH"] = str(MT) + os.pathsep + env.get("PYTHONPATH", "")
env["MPLBACKEND"] = "Agg"

cmd = [
    PY, "-m", "scripts.inference",
    "--inference_config", str(cfg),
    "--result_dir", str(result_dir),
    "--unet_model_path", "models/musetalkV15/unet.pth",
    "--unet_config", "models/musetalkV15/musetalk.json",
    "--whisper_dir", "models/whisper",
    "--version", "v15",
    "--fps", "25",
    "--batch_size", "2",
    "--parsing_mode", "jaw",
]

print("\nStarting MuseTalk 1.5 on T4...")
p = subprocess.run(cmd, cwd=str(MT), env=env, text=True, capture_output=True)
print(p.stdout)
print(p.stderr)
if p.returncode != 0:
    raise RuntimeError(f"MuseTalk inference failed with exit code {p.returncode}. Full stderr is printed above.")

candidates = list(result_dir.rglob("*.mp4"))
if not candidates:
    raise RuntimeError("MuseTalk exited successfully but produced no MP4.")
candidate = max(candidates, key=lambda x: x.stat().st_size)
if candidate.stat().st_size < 100_000:
    raise RuntimeError(f"Output MP4 is suspiciously small: {candidate}")
print(f"\nSUCCESS: {candidate}")
print(f"SIZE: {candidate.stat().st_size / 1024 / 1024:.2f} MB")


In [ ]:
# 6) Download the finished MP4
from google.colab import files
from pathlib import Path

result_dir = Path("/content/musetalk_output/result")
candidates = list(result_dir.rglob("*.mp4"))
if not candidates:
    raise RuntimeError("No final MP4 was found.")
candidate = max(candidates, key=lambda p: p.stat().st_size)
print("Final MP4:", candidate)
print("Size:", round(candidate.stat().st_size / 1024 / 1024, 2), "MB")
files.download(str(candidate))
